<a href="https://colab.research.google.com/github/antonypradeep54/Machine-Learning---Company-bankruptcy-prediction/blob/main/C4_M30_L2_Project_Company_Bankruptcy_Completed_on_06_Sep_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size=+3 color="blue"><center><b> Project : Company Bankruptcy</b></center></font>

## Table of contents

- Problem statement
- Libraries and functions
- Import dataset
- Exploratory Data Analysis (EDA)
    - Check for missing data
    - Correlation Analysis
    - Handle target class imbalance
- Model training and evaluation
    - Logistic Regression
    - K-Nearest Neighbors
    - Decision Tree
    - Neural Networks
    - Random Forest
    - Gradient Boosting
- Comparison of Results
- References

##Problem Statement
The goal is to build and evaluate machine learning models to predict company bankruptcy based on various financial indicators. The task involves addressing class imbalance in the dataset to improve model performance, particularly for predicting companies that are at risk of bankruptcy (minority class).

##Libraries and functions

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import os

##Import dataset

In [ ]:
upoaded = files.upload()

In [ ]:
print(os.listdir())

In [ ]:
# Move the uploaded bankruptcy dataset into the dataset/ folder.
os.makedirs('dataset', exist_ok=True)
import shutil
shutil.move('Dataset - Company Bankruptcy.csv', 'dataset/Dataset - Company Bankruptcy.csv')

In [ ]:
Company_data_df = pd.read_csv('dataset/Dataset - Company Bankruptcy.csv')
Company_data_df.head()

##Exploratory Data Analysis (EDA)

The first column "Bankrupt?" is our target variables (dependent variable) and other columns are features (independent variables).

In [ ]:
num_columns = Company_data_df.shape[1]
print(f"There are {num_columns} columns, lets explore if there are columns which we can drop due to missing data.")

####Check for missing data

In [ ]:
# Count missing values per column
missing_counts = Company_data_df.isnull().sum()

# Percentage of missing values
missing_percentage = (Company_data_df.isnull().mean() * 100).round(2)

# Combine into one DataFrame for clarity
missing_summary = pd.DataFrame({
    "Missing_Values": missing_counts,
    "Missing_Percentage": missing_percentage
}).sort_values(by="Missing_Percentage", ascending=False)


Count_columns_with_more_than_50_percent_missing_values = (missing_summary['Missing_Percentage'] > 50).sum()

print(f"There are {Count_columns_with_more_than_50_percent_missing_values} columns with more than 50% missing values, so we dont need to drop any columns.")

####Correlation Analysis

In [ ]:
# Explore correlations for descriptive analysis only; do not select features here.
corr_matrix = Company_data_df.corr(numeric_only=True)
target_corr = corr_matrix['Bankrupt?'].sort_values(ascending=False)
print(target_corr)

In [ ]:
# Feature selection is performed after the train/test split in the next section.
# This avoids using test-set information to choose columns.

In [ ]:
# See the training-only correlation filter below (threshold = 0.90).

In [ ]:
# Reduced features will be generated from the training partition only.
print('Original dataset shape:', Company_data_df.shape)

####Target variable class imbalance

In [ ]:
class_percentages = Company_data_df['Bankrupt?'].value_counts(normalize=True) * 100
class_percentages = class_percentages.round(2)


for cls, pct in class_percentages.items():
    print(f"Class {cls}: {pct:.2f}%")

print("\nThe dataset is highly imbalanced, because the minority class (Class 1) is severely underrepresented.")


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

Company_data_df_X = Company_data_df.drop('Bankrupt?', axis=1)
Company_data_df_y = Company_data_df['Bankrupt?']

Company_data_df_X_train, Company_data_df_X_test, Company_data_df_y_train, Company_data_df_y_test = train_test_split(
    Company_data_df_X, Company_data_df_y, test_size=0.2,
    stratify=Company_data_df_y, random_state=42
)

# Learn correlation-based feature removal from TRAINING FEATURES ONLY.
# Do not include the target or inspect test-set correlations.
train_corr = Company_data_df_X_train.corr(numeric_only=True).abs()
upper = train_corr.where(np.triu(np.ones(train_corr.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if (upper[column] > 0.90).any()]
keep_columns = Company_data_df_X_train.columns.drop(to_drop)

Company_data_reduced_X_train = Company_data_df_X_train.loc[:, keep_columns].copy()
Company_data_reduced_X_test = Company_data_df_X_test.loc[:, keep_columns].copy()
Company_data_reduced_y_train = Company_data_df_y_train.copy()
Company_data_reduced_y_test = Company_data_df_y_test.copy()

print('Features before:', Company_data_df_X_train.shape[1])
print('Features after:', Company_data_reduced_X_train.shape[1])
print('Removed correlated features:', len(to_drop))

In [ ]:
#Use SMOTE (Synthetic Minority Oversampling Technique) on original dataset
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
Company_data_df_X_train_SMOTE, Company_data_df_y_train_SMOTE = smote.fit_resample(Company_data_df_X_train, Company_data_df_y_train)

print("Before SMOTE:", Company_data_df_y_train.value_counts(normalize=True) * 100)
print("After SMOTE:", Company_data_df_y_train_SMOTE.value_counts(normalize=True) * 100)

In [ ]:
#Use SMOTE (Synthetic Minority Oversampling Technique) on reduced (coorelated columns dropped) dataset
smote = SMOTE(random_state=42)
Company_data_reduced_X_train_SMOTE, Company_data_reduced_y_train_SMOTE = smote.fit_resample(Company_data_reduced_X_train, Company_data_reduced_y_train)

print("Before SMOTE:", Company_data_reduced_y_train.value_counts(normalize=True) * 100)
print("After SMOTE:", Company_data_reduced_y_train_SMOTE.value_counts(normalize=True) * 100)
print("\n\nNow target class inbalance for both dataset (original and reduced) has been handeled using SMOTE.")

####Logistic Regression

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, f1_score, classification_report

# Train Logistic Regression with original dataset
lr_model_df = LogisticRegression(solver='liblinear', random_state=42)
lr_model_df.fit(Company_data_df_X_train_SMOTE, Company_data_df_y_train_SMOTE)

# Step 4: Make predictions
lr_model_df_y_pred = lr_model_df.predict(Company_data_df_X_test) #Company_data_df_y_test

# Step 5: Evaluate
lr_acc = accuracy_score(Company_data_df_y_test, lr_model_df_y_pred)
lr_prec = precision_score(Company_data_df_y_test, lr_model_df_y_pred)
lr_f1 = f1_score(Company_data_df_y_test, lr_model_df_y_pred)
lr_cm = confusion_matrix(Company_data_df_y_test, lr_model_df_y_pred)

# Print results
print("Logistics regression modle performance trained with original dataset")
print("----------------")
print("\nConfusion Matrix:\n", lr_cm)
print(f"\nAccuracy: {lr_acc:.2f}")
print(f"Precision: {lr_prec:.2f}")
print(f"F1-Score: {lr_f1:.2f}")


print('-----------------------------')

# Train Logistic Regression with reduced dataset
lr_model_reduced = LogisticRegression(solver='liblinear', random_state=42)
lr_model_reduced.fit(Company_data_reduced_X_train_SMOTE, Company_data_reduced_y_train_SMOTE)

# Make predictions
lr_model_reduced_y_pred = lr_model_reduced.predict(Company_data_reduced_X_test)

# Evaluate
lr_acc_reduced = accuracy_score(Company_data_reduced_y_test, lr_model_reduced_y_pred)
lr_prec_reduced = precision_score(Company_data_reduced_y_test, lr_model_reduced_y_pred)
lr_f1_reduced = f1_score(Company_data_reduced_y_test, lr_model_reduced_y_pred)
lr_cm_reduced = confusion_matrix(Company_data_reduced_y_test, lr_model_reduced_y_pred)

# Print results
print("\n\n\nLogistics regression modle performance trained with reduced dataset")
print("----------------")
print("\nConfusion Matrix:\n", lr_cm_reduced)
print(f"\nAccuracy: {lr_acc_reduced:.2f}")
print(f"Precision: {lr_prec_reduced:.2f}")
print(f"F1-Score: {lr_f1_reduced:.2f}")

Model_score = []
Model_score.append(("Logistics Regression", lr_acc_reduced, lr_prec_reduced,lr_f1_reduced,lr_cm_reduced))
print("\n\n\nWe can notice that both models lr_model_df trained with original dataset and lr_model_reduced trained with reduced dataset perfroms the same. \nFor future models we will be using the reduced dataset to minimize computation effort.")


Confusion Matrix:
 [[1019  301]
 [  28   16]]

Accuracy: 0.76
Precision: 0.05
F1-Score: 0.09

####K-Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier


# Train KNN with reduced dataset
knn_model_reduced = KNeighborsClassifier(n_neighbors=5)
knn_model_reduced.fit(Company_data_reduced_X_train_SMOTE, Company_data_reduced_y_train_SMOTE)

knn_model_reduced_y_pred = knn_model_reduced.predict(Company_data_reduced_X_test)

# Evaluate
knn_acc_reduced = accuracy_score(Company_data_reduced_y_test, knn_model_reduced_y_pred)
knn_prec_reduced = precision_score(Company_data_reduced_y_test, knn_model_reduced_y_pred)
knn_f1_reduced = f1_score(Company_data_reduced_y_test, knn_model_reduced_y_pred)
knn_cm_reduced = confusion_matrix(Company_data_reduced_y_test, knn_model_reduced_y_pred)

# Print results
print("\n\n\nk-nearest neighbors (KNN) modle performance trained with reduced dataset")
print("\nConfusion Matrix:\n", knn_cm_reduced)
print(f"\nAccuracy: {knn_acc_reduced:.2f}")
print(f"Precision: {knn_prec_reduced:.2f}")
print(f"F1-Score: {knn_f1_reduced:.2f}")

Model_score.append(("K-Nearest Neighbors",knn_acc_reduced, knn_prec_reduced,knn_f1_reduced,knn_cm_reduced))

####Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Train Decision Tree Classifier with reduced dataset
dtc_model_reduced = DecisionTreeClassifier(random_state=42)
dtc_model_reduced.fit(Company_data_reduced_X_train_SMOTE, Company_data_reduced_y_train_SMOTE)

dtc_model_reduced_y_pred = dtc_model_reduced.predict(Company_data_reduced_X_test)

# Evaluate
dtc_acc_reduced = accuracy_score(Company_data_reduced_y_test, dtc_model_reduced_y_pred)
dtc_prec_reduced = precision_score(Company_data_reduced_y_test, dtc_model_reduced_y_pred)
dtc_f1_reduced = f1_score(Company_data_reduced_y_test, dtc_model_reduced_y_pred)
dtc_cm_reduced = confusion_matrix(Company_data_reduced_y_test, dtc_model_reduced_y_pred)

# Print results
print("\n\n\nDecision Tree Classifier modle performance trained with reduced dataset")
print("\nConfusion Matrix:\n", dtc_cm_reduced)
print(f"\nAccuracy: {dtc_acc_reduced:.2f}")
print(f"Precision: {dtc_prec_reduced:.2f}")
print(f"F1-Score: {dtc_f1_reduced:.2f}")

Model_score.append(("Decision Tree Classifier",dtc_acc_reduced, dtc_prec_reduced,dtc_f1_reduced,dtc_cm_reduced))

####Neural Networks

In [ ]:
from sklearn.neural_network import MLPClassifier

# Train Multi-Layer Perceptron with reduced dataset
mlp_model_reduced = MLPClassifier(
    hidden_layer_sizes=(64, 32),  # two hidden layers with 64 and 32 neurons
    activation='relu',            # common activation function
    solver='adam',                # optimizer
    max_iter=500,                 # increase iterations for convergence
    random_state=42
)
mlp_model_reduced.fit(Company_data_reduced_X_train_SMOTE, Company_data_reduced_y_train_SMOTE)

mlp_model_reduced_y_pred = mlp_model_reduced.predict(Company_data_reduced_X_test)

# Evaluate
mlp_acc_reduced = accuracy_score(Company_data_reduced_y_test, mlp_model_reduced_y_pred)
mlp_prec_reduced = precision_score(Company_data_reduced_y_test, mlp_model_reduced_y_pred)
mlp_f1_reduced = f1_score(Company_data_reduced_y_test, mlp_model_reduced_y_pred)
mlp_cm_reduced = confusion_matrix(Company_data_reduced_y_test, mlp_model_reduced_y_pred)

# Print results
print("\n\n\nMulti-Layer Perceptron modle performance trained with reduced dataset")
print("\nConfusion Matrix:\n", mlp_cm_reduced)
print(f"\nAccuracy: {mlp_acc_reduced:.2f}")
print(f"Precision: {mlp_prec_reduced:.2f}")
print(f"F1-Score: {mlp_f1_reduced:.2f}")

Model_score.append(("Multi-Layer Perceptron",mlp_acc_reduced, mlp_prec_reduced,mlp_f1_reduced,mlp_cm_reduced))

####Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest Classifier with reduced dataset
rfc_model_reduced = RandomForestClassifier(
    n_estimators=100,   # number of trees
    max_depth=None,     # let trees expand fully (can tune later)
    random_state=42
)
rfc_model_reduced.fit(Company_data_reduced_X_train_SMOTE, Company_data_reduced_y_train_SMOTE)

rfc_model_reduced_y_pred = rfc_model_reduced.predict(Company_data_reduced_X_test)

# Evaluate
rfc_acc_reduced = accuracy_score(Company_data_reduced_y_test, rfc_model_reduced_y_pred)
rfc_prec_reduced = precision_score(Company_data_reduced_y_test, rfc_model_reduced_y_pred)
rfc_f1_reduced = f1_score(Company_data_reduced_y_test, rfc_model_reduced_y_pred)
rfc_cm_reduced = confusion_matrix(Company_data_reduced_y_test, rfc_model_reduced_y_pred)

# Print results
print("\n\n\nRandom Forest Classifier modle performance trained with reduced dataset")
print("\nConfusion Matrix:\n", rfc_cm_reduced)
print(f"\nAccuracy: {rfc_acc_reduced:.2f}")
print(f"Precision: {rfc_prec_reduced:.2f}")
print(f"F1-Score: {rfc_f1_reduced:.2f}")

Model_score.append(("Random Forest Classifier",rfc_acc_reduced, rfc_prec_reduced,rfc_f1_reduced,rfc_cm_reduced))

####

####Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Train Gradient Boosting Classifier with reduced dataset
gb_model_reduced = GradientBoostingClassifier(
    n_estimators=200,   # number of boosting stages (trees)
    learning_rate=0.1,  # step size (smaller = more robust, slower)
    max_depth=3,        # depth of each tree
    random_state=42
)

gb_model_reduced.fit(Company_data_reduced_X_train_SMOTE, Company_data_reduced_y_train_SMOTE)

gb_model_reduced_y_pred = gb_model_reduced.predict(Company_data_reduced_X_test)

# Evaluate
gb_acc_reduced = accuracy_score(Company_data_reduced_y_test, gb_model_reduced_y_pred)
gb_prec_reduced = precision_score(Company_data_reduced_y_test, gb_model_reduced_y_pred)
gb_f1_reduced = f1_score(Company_data_reduced_y_test, gb_model_reduced_y_pred)
gb_cm_reduced = confusion_matrix(Company_data_reduced_y_test, gb_model_reduced_y_pred)

# Print results
print("\n\n\nGradient Boosting Classifier modle performance trained with reduced dataset")
print("\nConfusion Matrix:\n", gb_cm_reduced)
print(f"\nAccuracy: {gb_acc_reduced:.2f}")
print(f"Precision: {gb_prec_reduced:.2f}")
print(f"F1-Score: {gb_f1_reduced:.2f}")

Model_score.append(("Gradient Boosting Classifier",gb_acc_reduced, gb_prec_reduced,gb_f1_reduced,gb_cm_reduced))

####Comparison of Results

In [ ]:
Model_Score_df = pd.DataFrame(Model_score, columns=["Model", "Accuracy", "Precision", "F1 Score", "Confusion Matrix"])

In [ ]:
Model_Score_df.style.format({
    "Accuracy": "{:.2f}",
    "Precision": "{:.2f}",
    "F1 Score": "{:.2f}"
})

####Conclusion

In [ ]:
print("We can notice that Random Forest Classifier and Gradient Boosting Classifier are better perfroming models")
print(f"\nRandom Forest Classifier has an accuracy of {rfc_acc_reduced:.2f}, precision of {rfc_prec_reduced:.2f}, and F1 score of {rfc_f1_reduced:.2f} \nGradient Boosting Classifier has an accuracy of {gb_acc_reduced:.2f}, precision of {gb_prec_reduced:.2f}, and F1 score of {gb_f1_reduced:.2f}.")

####References

  - 5 Minutes Engineering youtube channel for machine learning algorithns.
  - ChatGPT for cooding assistance.

<font size=+2 color="blue"><center><b> End of project</b></center></font>